In [1]:
# IMPORTS
import sys
import os
import torch
import torch.nn as nn
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime as dt

import pandas, anndata, torch
print(pandas.__version__, anndata.__version__, torch.__version__)

# IMPORT OWN FILES
sys.path.append(os.path.abspath(".."))
import data_handling as dh
import binn_training as bt
from binn import BINN

# Supress warnings
import warnings
warnings.filterwarnings("ignore", message="Series.__getitem__ treating keys as positions", category=FutureWarning)
warnings.filterwarnings("ignore", message="The class AnnLoader is deprecated", category=FutureWarning)

C:\Users\nordm\AppData\Local\Temp\ipykernel_1572\1997014344.py:14: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print(pandas.__version__, anndata.__version__, torch.__version__)


2.3.3 0.13.4 2.14.1+cpu


In [2]:
import importlib.util
from pathlib import Path

# Determine parent folder
base_dir = Path.cwd().parent
path = base_dir / "pipeline_paths_mnt.py"

# Import module dynamically
spec = importlib.util.spec_from_file_location("ppaths", path)
ppaths = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ppaths)

# to keep consistent paths
run_vars = "mg_200_mc_200_mhvg1000"
pp = ppaths.PipelinePaths("dev_local", full_pipeline_run_vars=run_vars)

base path found (D:\BBTX11-VT26-07\simulated): True
data path found (D:\BBTX11-VT26-07\simulated\data): True


In [5]:
# GLOBALS
TRAIN_SIZE = 0.8
BATCH_SIZE = 32
EPOCHS = 20
TUNE_EPOCHS = 10
FOLDS_OUTER = 3
FOLDS_INNER = 3
EVAL_METRIC = 'accuracy'

# PARAMETERS
# OPTUNA: {'lr': 0.002204119007996014, 'weight_decay': 0.021469124496437196, 'dropout': 0.17634528194149005}
LR = 0.002204
WEIGHT_DECAY = 0.021469
DROPOUT = 0.176345
ACTIVATION_FN = nn.Tanh()

# PATHS
data_path = pp.compl_full_pipe_path

# OTHER
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TUNE_HYPERPARAMETERS = False

now = dt.now().strftime("%y%m%d_%H%M")
if TUNE_HYPERPARAMETERS:
    now_hyper = now
else:
    # hyperparameters to read
    now_hyper = '261110_0958'

cell_types = ['test1']

all_results = {}
for cell_type in cell_types:
    print(f'RUNNING BINN PIPELINE FOR {cell_type}')
    data_info = f'{cell_type}_{run_vars}'
    HPARAM_PATH = pp.binn_model_path / f"{now_hyper}_best_hyperparameters_{data_info}.json"
    MASK_PATHS = [pp.mask_full_pipe_path / run_vars / f'{cell_type}_train_preprocessed_layer_{n}_mask.csv' 
              for n in range(5)]

    print("Reading masks...")
    masks = dh.read_masks(MASK_PATHS, print_shapes=True)

    print("Reading train and test data...")
    train_adata = ad.read_h5ad(data_path / f'{cell_type}_train_preprocessed.h5ad')
    test_adata = ad.read_h5ad(data_path / f'{cell_type}_test_preprocessed.h5ad')

    print(f'train_adata:\n{train_adata}')

    print("Roll-up to patient level...")
    patient_datasets = dh.rollup_to_patient_level(
        {'test':test_adata, 'train':train_adata}
        )

    print("Aligning adatas to BINN...")
    datasets_aligned = dh.subset_genes(patient_datasets, masks['df0'])

    print("Padding adatas to BINN-ready shape...")
    datasets_padded = dh.pad_align_data(datasets_aligned, masks["df0"])

    print("Getting dataloaders...")
    train_loader_global, test_loader_global = dh.create_dataloaders(
        datasets_padded['train'], 
        datasets_padded['test'], 
        batch_size=BATCH_SIZE
        )

    mask_matrix_list, in_features, layers_list, tensor_masks = dh.compute_features(masks, device)

    dh.poison_scanner(train_loader_global, device)
    dh.poison_scanner(test_loader_global, device)

    # Hyperparameter tuning, takes a while 
    if TUNE_HYPERPARAMETERS:
        print("Performing hyperparameter tuning...")
        best_params = dh.hyperparameter_tuning_optuna_mean_curve(
            datasets_padded['train'], 
            in_features, 
            layers_list, 
            tensor_masks, 
            device, 
            batch_size=BATCH_SIZE, 
            activation_fn=ACTIVATION_FN,
            k=FOLDS_INNER, 
            epochs=TUNE_EPOCHS,
            eval_metric=EVAL_METRIC
            )
        print(f"Best parameters for BINN: {best_params}")
        dh.save_hyperparameters(best_params, HPARAM_PATH)

    hparams = dh.load_hyperparameters(
        HPARAM_PATH,
            {"lr": LR, 
            "weight_decay": WEIGHT_DECAY, 
            "dropout": DROPOUT,
            "epochs": None,
            "tune_total_epochs": None
            }
        )

    # CROSS VALIDATION
    # select one shared epoch from the mean curve across folds
    all_fold_scores, cv_best_epoch = dh.run_CV_nested(
        datasets_padded['train'], 
        in_features, 
        layers_list, 
        tensor_masks,                                       
        device,
        eval_metric=EVAL_METRIC,
        k_outer=FOLDS_OUTER,
        k_inner=FOLDS_INNER,
        epochs=EPOCHS, 
        lr=hparams["lr"], 
        weight_decay=hparams["weight_decay"], 
        dropout=hparams["dropout"],
        activation_fn=ACTIVATION_FN,
        save_path=pp.binn_model_path,
        data_info=data_info,
        time=now
        )


RUNNING BINN PIPELINE FOR test1
Reading masks...
Matrix 0 shape: (939, 793)
Matrix 1 shape: (793, 450)
Matrix 2 shape: (450, 168)
Matrix 3 shape: (168, 29)
Matrix 4 shape: (29, 1)
Reading train and test data...


d:\BBTX11-VT26-07\venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
[I 2026-11-17 17:22:15,578] A new study created in memory with name: no-name-c7c17415-e5ca-48c2-b2dc-867384aea721


train_adata:
AnnData object with n_obs × n_vars = 6 × 1000
    obs: 'n_obs_aggregated', 'msex', 'age_death', 'pmi', 'race', 'Pathologic_diagnosis_of_AD', 'cogdx', 'braaksc', 'ceradsc', 'label', 'n_genes', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito'
    var: 'n_cells', 'mito', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    layers: 'mean', None (.X)
Roll-up to patient level...
Rolling up 'test'...
Rolling up 'train'...
Aligning adatas to BINN...
Overlapping genes kept: 939 for test
Overlapping genes kept: 939 for train
Padding adatas to BINN-ready shape...
Getting dataloaders...
input features: 939
layer list: [793, 450, 168, 29, 1]
No poison found!
No poison found!
Starting Optuna Hype

d:\BBTX11-VT26-07\venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
[I 2026-11-17 17:22:17,476] Trial 0 finished with value: 0.5 and parameters: {'lr': 1.547334446495307e-05, 'weight_decay': 0.02230754846720165, 'dropout': 0.3734351692292374}. Best is trial 0 with value: 0.5.
d:\BBTX11-VT26-07\venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
[I 2026-11-17 17:22:19,054] Trial 1 finished with value: 0.8333333333333334 and parameters: {'lr': 0.0054076540927755126, 'weight_decay': 0.00026319591800609435, 'dropout': 0.2505641855056806}. Best is trial 1 with value: 0.8333333333333334.
d:\BBTX11-VT26-07\venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_

Best accuracy: 0.8333 at epoch 1, params={'lr': 0.0054076540927755126, 'weight_decay': 0.00026319591800609435, 'dropout': 0.2505641855056806}
Outer fold 1: score accuracy = 1.0000, params={'lr': 0.0054076540927755126, 'weight_decay': 0.00026319591800609435, 'dropout': 0.2505641855056806, 'epochs': 1, 'tune_total_epochs': 20, 'metric': 'accuracy', 'cv_score': 0.8333333333333334}
Starting Optuna Hyperparameter Search...


[I 2026-11-17 17:22:33,028] Trial 0 finished with value: 0.8333333333333334 and parameters: {'lr': 0.002945751585187655, 'weight_decay': 0.01785133623021421, 'dropout': 0.4751196746818946}. Best is trial 0 with value: 0.8333333333333334.
d:\BBTX11-VT26-07\venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
[I 2026-11-17 17:22:34,792] Trial 1 finished with value: 0.3333333333333333 and parameters: {'lr': 0.00013667661664952454, 'weight_decay': 0.00041886090784119736, 'dropout': 0.224785841222906}. Best is trial 0 with value: 0.8333333333333334.
d:\BBTX11-VT26-07\venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
[I 2026-11-17 17:22:36,327] Trial 2 finished with value: 0.8333333333333334 and parameters: {'lr': 0.003634738502347696, 'weight_decay':

Best accuracy: 1.0000 at epoch 1, params={'lr': 1.1427522120810294e-05, 'weight_decay': 0.0007154539923819097, 'dropout': 0.36190011947187845}
Outer fold 2: score accuracy = 0.5000, params={'lr': 1.1427522120810294e-05, 'weight_decay': 0.0007154539923819097, 'dropout': 0.36190011947187845, 'epochs': 1, 'tune_total_epochs': 20, 'metric': 'accuracy', 'cv_score': 1.0}
Starting Optuna Hyperparameter Search...


[I 2026-11-17 17:22:49,251] Trial 0 finished with value: 1.0 and parameters: {'lr': 0.0015756323791994925, 'weight_decay': 0.00019235303596685233, 'dropout': 0.4122670528839376}. Best is trial 0 with value: 1.0.
[I 2026-11-17 17:22:50,784] Trial 1 finished with value: 1.0 and parameters: {'lr': 0.0038053491065524053, 'weight_decay': 0.009932085353294706, 'dropout': 0.13774980787223137}. Best is trial 0 with value: 1.0.
[I 2026-11-17 17:22:52,417] Trial 2 finished with value: 0.6666666666666666 and parameters: {'lr': 7.480508367871875e-05, 'weight_decay': 0.09926807306552934, 'dropout': 0.20819717076282387}. Best is trial 0 with value: 1.0.
[I 2026-11-17 17:22:53,977] Trial 3 finished with value: 0.16666666666666666 and parameters: {'lr': 3.1158063573920104e-05, 'weight_decay': 0.0012547780336687848, 'dropout': 0.3336932349799886}. Best is trial 0 with value: 1.0.
[I 2026-11-17 17:22:55,472] Trial 4 finished with value: 1.0 and parameters: {'lr': 0.00012209042499690574, 'weight_decay': 

Best accuracy: 1.0000 at epoch 2, params={'lr': 0.0015756323791994925, 'weight_decay': 0.00019235303596685233, 'dropout': 0.4122670528839376}
Outer fold 3: score accuracy = 0.5000, params={'lr': 0.0015756323791994925, 'weight_decay': 0.00019235303596685233, 'dropout': 0.4122670528839376, 'epochs': 2, 'tune_total_epochs': 20, 'metric': 'accuracy', 'cv_score': 1.0}
Nested CV Score (accuracy): 0.6667 +/- 0.2357


In [ ]:

    # model is MLP with same layers as BINN 
    binn, criterion, optimizer, scheduler = dh.create_model(
        in_features, 
        layers_list, 
        tensor_masks, 
        device, 
        lr=hparams["lr"], 
        weight_decay=hparams["weight_decay"], 
        dropout=hparams["dropout"], 
        epochs=EPOCHS,
        activation_fn=ACTIVATION_FN
        )
    print(binn)

    print(f"criterion: {criterion},\n optimizer: {optimizer}")

    # Run it
    dead_nodes_dict = dh.find_dead_outputs(binn, tensor_masks)
    print(dead_nodes_dict)

    # Dead output
    dh.show_dead_nodes(dead_nodes_dict, masks)

    for batch in train_loader_global:
        print(batch.X)
        break

    history = dh.train_fixed_epochs(
            binn, 
            train_loader_global, 
            criterion=criterion, 
            optimizer=optimizer, 
            scheduler=scheduler, 
            device=device,
            n_epochs=cv_best_epoch,
            save_path=pp.binn_model_path,
            data_info=data_info,
            time=now,
            monitor_loader=test_loader_global
            )

    # Harvest the results
    probs, targets, test_auc = dh.evaluate_model_roc(binn, test_loader_global, device)
    print(f"Test ROC-AUC: {test_auc:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    # Loss plot
    axes[0].plot(history['train_loss'], label='Training Loss', color='royalblue', linewidth=2)
    axes[0].plot(history['test_loss'], label='Testing Loss', color='orange', linestyle='--', linewidth=2)
    axes[0].set_title('Loss', fontsize=14)
    axes[0].set_xlabel('Epochs', fontsize=12)
    axes[0].set_ylabel('Loss (BCE)', fontsize=12)
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Accuracy plot
    axes[1].plot(history['train_acc'], label='Training Accuracy', color='green', linewidth=2)
    axes[1].plot(history['test_acc'], label='Testing Accuracy', color='red', linestyle='--', linewidth=2)
    axes[1].set_title('Accuracy', fontsize=14)
    axes[1].set_xlabel('Epochs', fontsize=12)
    axes[1].set_ylabel('Accuracy', fontsize=12)
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle('BINN Training Diagnostics', fontsize=16)
    plt.tight_layout()
    fig.savefig(pp.binn_test_results_path / f'{now}_training_curves_{data_info}.png')
    plt.show()

    df_res = dh.save_test_results(
        binn, 
        test_loader_global, 
        device, 
        data_info, 
        pp.binn_test_results_path,
        now
        )
    
    dh.confusion_matrix_binn(df_res, pp.binn_test_results_path, data_info, now)

    report = dh.class_report(df_res)
    print(report)

    all_results[cell_type] = {
        'cv_auc_mean': np.mean(all_fold_scores), 
        'cv_auc_std': np.std(all_fold_scores),
        'cv_best_epoch': cv_best_epoch, 
        'cv_total_epochs': EPOCHS,
        'test_auc': test_auc,
        **{f'hp_{k}': v for k, v in hparams.items()
        },
    }
    
    pd.DataFrame(history).to_csv(
        pp.binn_model_path/ f'{now}_history_{data_info}.csv', 
        index_label='epoch')
    
    with open(pp.binn_test_results_path / f'{now}_class_report_{data_info}.txt', 'w') as f:
        f.write(report)

    print(f'COMPLETED BINN PIPELINE FOR {cell_type} \n\n\n')

summary = pd.DataFrame(all_results).T
summary.to_csv(pp.binn_test_results_path / f'{now}_summary_{run_vars}.csv')
summary

